## Setup Models

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os

# GOOGLE_API_KEY is set as a Codespace secret

model = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash"

)


## 06.03. Using agents as Graph nodes

In [ ]:
%run "code_03_XX Product QnA Agentic chatbot (1).ipynb"
print("===============================================================")
%run "code_04_XX Orders Chatbot with custom agent (1).ipynb"


In [ ]:
import functools
# Helper function to invoke an agent
def agent_node(state, agent, name, config):

    #extract thread-id from request for conversation memory
    thread_id=config["metadata"]["thread_id"]
    #Set the config for calling the agent
    agent_config = {"configurable": {"thread_id": thread_id}}

    #Pass the thread-id to establish memory for chatbot
    #Invoke the agent with the state
    result = agent.invoke(state, agent_config)

    # Convert the agent output into a format that is suitable to append to the global state
    if isinstance(result, ToolMessage):
        pass
    else:
        final_result=AIMessage(result['messages'][-1].content)
    return {
        "messages": [final_result]
    }

#Create the product QnA node
product_QnA_node=functools.partial(agent_node, 
                                   agent=product_QnA_agent, 
                                   name="Product_QnA_Agent")
#Create the Orders node
orders_node=functools.partial(agent_node,
                              agent=orders_agent.agent_graph,
                              name="Orders_Agent")

#Create the Refund node
refund_node=functools.partial(agent_node,
                              agent=refund_agent.agent_graph,
                              name="Refund_Agent")


## 06.04. Create the Routing Agent & Chatbot

In [ ]:
#Creating the routing agent

from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
import operator

class RouterAgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

class RouterAgent:

    def __init__(self, model, system_prompt, smalltalk_prompt, debug=False):
        
        self.system_prompt = system_prompt
        self.smalltalk_prompt = smalltalk_prompt
        self.model = model
        self.debug = debug
        
        router_graph = StateGraph(RouterAgentState)
        router_graph.add_node("Router", self.call_llm)
        router_graph.add_node("Product_Agent", product_QnA_node)
        router_graph.add_node("Orders_Agent", orders_node)
        router_graph.add_node("Refund_Agent", refund_node)
        router_graph.add_node("Small_Talk", self.respond_smalltalk)
                              
        router_graph.add_conditional_edges(
            "Router",
            self.find_route,
            {"PRODUCT": "Product_Agent", 
             "ORDER" : "Orders_Agent",
             "REFUND" : "Refund_Agent",
             "SMALLTALK" : "Small_Talk",
             "END": END }
        )

        router_graph.add_edge("Product_Agent", END)
        router_graph.add_edge("Orders_Agent", END)
        router_graph.add_edge("Refund_Agent", END)
        router_graph.add_edge("Small_Talk", END)
        
        router_graph.set_entry_point("Router")
        self.router_graph = router_graph.compile()

    def call_llm(self, state: RouterAgentState):
        messages = state["messages"]
        if self.debug:
            print(f"Call LLM received {messages}")
            
        if self.system_prompt:
            messages = [SystemMessage(content=self.system_prompt)] + messages

        result = self.model.invoke(messages)

        if self.debug:
            print(f"Call LLM result {result}")
        return {"messages": [result]}

    def respond_smalltalk(self, state: RouterAgentState):
        messages = state["messages"]
        if self.debug:
            print(f"Small talk received: {messages}")
            
        messages = [SystemMessage(content=self.smalltalk_prompt)] + messages
        result = self.model.invoke(messages)

        if self.debug:
            print(f"Small talk result {result}")
        return {"messages": [result]}
        
    def find_route(self, state: RouterAgentState):
        last_message = state["messages"][-1]
        if self.debug: 
            print("Router: Last result from LLM : ", last_message)

        destination = last_message.content.strip()

        if self.debug:
            print(f"Destination chosen : {destination}")
        return destination


In [ ]:
#Create the chatbot
from IPython.display import Image

system_prompt = """ 
You are a Router that analyzes the input query and chooses one of 5 options:
SMALLTALK: If the user input is small talk, like greetings and good byes.
PRODUCT: If the query is about golf products — features, specs, pricing, or recommendations.
ORDER: If the query is about orders — order status, order details, or updating an order.
REFUND: If the query is about refunds, returns, exchanges, cancellations, or store policies.
END: Default, when it is none of the above.

The output should only be just one word out of the possible 5: SMALLTALK, PRODUCT, ORDER, REFUND, END.
"""

smalltalk_prompt = """
You are the front desk AI at Golf Gear Pro — think HAL 9000 if he ran a pro shop.
You are polite and helpful on the surface, but you have a dry, slightly condescending wit.
When greeting customers, be cordial but subtly imply you already know their handicap
is higher than they claim. Mention you can help with golf product info, order status,
and refund/return policies. Keep it concise.
"""

router_agent = RouterAgent(model, 
                           system_prompt, 
                           smalltalk_prompt,
                           debug=False)

Image(router_agent.router_graph.get_graph().draw_mermaid_png())


## 06.05 Execute the Routing chatbot

In [ ]:
#Execute a single request
import uuid
config = {"configurable": {"thread_id": str(uuid.uuid4())}}

messages=[HumanMessage(content="Tell me about the StormDrive Driver")]
result=router_agent.router_graph.invoke({"messages":messages},config)
for message in result['messages']:
    print(message.pretty_repr())


In [ ]:
#Execute a single request
messages=[HumanMessage(content="What is the status of order G1002?")]
result=router_agent.router_graph.invoke({"messages":messages},config)
for message in result['messages']:
    print(message.pretty_repr())


In [ ]:
import uuid
from langchain_core.messages import HumanMessage

#Send a sequence of messages to chatbot and get its response
user_inputs = [
    "Hello!",                                                # Turn 1: smalltalk
    "What golf products do you carry?",                      # Turn 2: product agent
    "Tell me about the FairwayPro Iron Set",                 # Turn 3: product agent
    "How much does it cost?",                                # Turn 4: product (context preserved)
    "Show me order G1001",                                   # Turn 5: order agent
    "What is your refund policy for damaged items?",         # Turn 6: refund agent
    "Thanks, bye!"                                           # Turn 7: smalltalk
]

#Create a new thread
config = {"configurable": {"thread_id": str(uuid.uuid4())}}

for user_input in user_inputs:
    print(f"----------------------------------------\nUSER : {user_input}")
    user_message = {"messages":[HumanMessage(user_input)]}
    ai_response = router_agent.router_graph.invoke(user_message, config=config)
    print(f"\nAGENT : {ai_response['messages'][-1].content}")
